# Sauvegarder ce qui permet de reprendre

Conserver le modèle et l’historique nécessaire à l’optimiseur.

Nous exécutons les cellules dans l’ordre, sur **CPU**. Aucun téléchargement de données ou de modèle n’est nécessaire.

Version de référence : **Python 3.11, PyTorch 2.10.0+cpu**. La cellule suivante indique la version réellement utilisée. Les sorties peuvent varier avec une autre version.

[Relire le chapitre](https://radiocodeur.github.io/quiz/cours.html?theme=pytorch#sauvegarder-et-reprendre). Les deux exercices sont des textes à compléter : ils n’interrompent pas l’exécution intégrale. Leurs solutions restent repliées à la fin.

In [1]:
import platform
import torch

torch.set_num_threads(1)
torch.manual_seed(17)
print("Python", platform.python_version())
print("PyTorch", torch.__version__)
print("Calcul", "CPU", "— threads", torch.get_num_threads())

Python 3.11.9
PyTorch 2.10.0+cpu
Calcul CPU — threads 1


## Nos repères

- **Sauvegarde (checkpoint)** : Les états et repères nécessaires à une reprise définie.
- **Dictionnaire d’état (state_dict)** : Les valeurs nommées enregistrées par un modèle ou un optimiseur.
- **État d’optimiseur (optimizer state)** : L’historique utilisé par sa règle de mise à jour.

## Le calcul du cours

Avant de lancer cette cellule, annonçons les entrées et la sortie attendue. Après l’exécution, expliquons les nombres et les formes affichés. Le programme est identique à celui du chapitre.

In [2]:
import tempfile
from pathlib import Path
import torch
torch.manual_seed(17)
model = torch.nn.Linear(1, 1, bias=False)
with torch.no_grad():
    model.weight.fill_(1.)
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)
optimizer.zero_grad()
model(torch.ones(1, 1)).square().mean().backward()
optimizer.step()
with tempfile.TemporaryDirectory() as dossier:
    fichier = Path(dossier) / "reprise.pt"
    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "epoch": 1}, fichier)
    etat = torch.load(fichier, map_location="cpu", weights_only=True)
    copie = torch.nn.Linear(1, 1, bias=False)
    nouvel_optim = torch.optim.Adam(copie.parameters(), lr=0.1)
    copie.load_state_dict(etat["model"])
    nouvel_optim.load_state_dict(etat["optimizer"])
print("poids", round(copie.weight.item(), 6))
print("identique", torch.equal(model.weight, copie.weight))
print("etats_adam", len(nouvel_optim.state), "epoch", etat["epoch"])

poids 0.9
identique True
etats_adam 1 epoch 1


## Nous vérifions notre compréhension

À dire en entretien : Je sauvegarde les poids et la préparation pour prédire. Pour reprendre l’apprentissage, je conserve aussi l’optimiseur et les autres états actifs.

## Exercice 1 — Retrouver les mêmes prédictions

Sauvegardons les poids, puis chargeons-les dans un modèle de même architecture.

Remplaçons les points de suspension. Nous pouvons copier ce texte dans une nouvelle cellule de code.

```python
import tempfile
from pathlib import Path
model = torch.nn.Linear(1, 1)
with tempfile.TemporaryDirectory() as dossier:
    fichier = Path(dossier) / "poids.pt"
    torch.save(..., fichier)
    copie = torch.nn.Linear(1, 1)
    copie.load_state_dict(torch.load(fichier, map_location="cpu", weights_only=True))
print(torch.equal(model(torch.ones(1, 1)), copie(torch.ones(1, 1))))
```

## Exercice 2 — Retrouver aussi l’état de l’optimiseur

Après un premier pas Adam, sauvegardons son état avec celui du modèle et le numéro d’époque.

Remplaçons les points de suspension. Nous pouvons copier ce texte dans une nouvelle cellule de code.

```python
model = torch.nn.Linear(1, 1)
optim = torch.optim.Adam(model.parameters(), lr=.1)
optim.zero_grad()
model(torch.ones(1, 1)).square().mean().backward()
optim.step()
etat = {"model": ..., "optimizer": ..., "epoch": 1}
copie = torch.nn.Linear(1, 1)
nouvel_optim = torch.optim.Adam(copie.parameters(), lr=.1)
copie.load_state_dict(etat["model"])
nouvel_optim.load_state_dict(...)
print(len(nouvel_optim.state), etat["epoch"])
```

# Solutions

Nous cherchons d’abord, puis ouvrons la solution utile. Chaque programme ci-dessous est autonome. Le vérificateur du dépôt exécute également ces solutions, séparément des exercices à trous.

<details>
<summary>Afficher la solution de l’exercice 1 — Retrouver les mêmes prédictions</summary>

<pre><code>import tempfile
from pathlib import Path
import torch
torch.manual_seed(17)
model = torch.nn.Linear(1, 1)
with tempfile.TemporaryDirectory() as dossier:
    fichier = Path(dossier) / &quot;poids.pt&quot;
    torch.save(model.state_dict(), fichier)
    copie = torch.nn.Linear(1, 1)
    copie.load_state_dict(torch.load(fichier, map_location=&quot;cpu&quot;, weights_only=True))
identique = torch.equal(model(torch.ones(1, 1)), copie(torch.ones(1, 1)))
assert identique
print(identique)</code></pre>
</details>

<details>
<summary>Afficher la solution de l’exercice 2 — Retrouver aussi l’état de l’optimiseur</summary>

<pre><code>import torch
torch.manual_seed(17)
model = torch.nn.Linear(1, 1)
optim = torch.optim.Adam(model.parameters(), lr=.1)
optim.zero_grad()
model(torch.ones(1, 1)).square().mean().backward()
optim.step()
etat = {&quot;model&quot;: model.state_dict(), &quot;optimizer&quot;: optim.state_dict(), &quot;epoch&quot;: 1}
copie = torch.nn.Linear(1, 1)
nouvel_optim = torch.optim.Adam(copie.parameters(), lr=.1)
copie.load_state_dict(etat[&quot;model&quot;])
nouvel_optim.load_state_dict(etat[&quot;optimizer&quot;])
assert len(nouvel_optim.state) == len(list(copie.parameters()))
assert torch.equal(model(torch.ones(1, 1)), copie(torch.ones(1, 1)))
print(len(nouvel_optim.state), etat[&quot;epoch&quot;])</code></pre>
</details>